# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print("\nKeywords:", metadata.keywords)
print("\nAuthors (by @id):")
for author in metadata.author:
    print(author['@id'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id` fields, as per Croissant standard.

In [ ]:
# Show available record sets and their @id
record_sets = dataset.record_sets
print("Available RecordSets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']} (Name: {rs.get('name', 'N/A')})")
    print("  Fields/columns:")
    for field in rs.get('fields', []):
        print(f"    - {field['@id']} (Name: {field.get('name', 'N/A')})")

# Preview first three records from the first available record set
if record_sets:
first_record_set_id = record_sets[0]['@id']
print(f"\nSample records from RecordSet {first_record_set_id}:")
for i, rec in enumerate(dataset.records(record_set=first_record_set_id)):
    print(rec)
    if i >= 2:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview above.

In [ ]:
# List all record set @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Extract data into pandas DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Show info for the main survey record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns for RecordSet {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
    print("\nPreview:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data using the proper `@id` for columns.

For demonstration, let's filter by GAD-7 score (field `@id`, normally referred to as `gad7_score`), normalize it, and group by `level_of_education` (field `@id`).

In [ ]:
# Example field @ids (please replace with actual IDs as needed)
numeric_field_id = 'gad7_score'  # Example field @id for GAD-7 Score
group_field_id = 'level_of_education'  # Example field @id for education

df = dataframes[main_record_set_id]

# Filter records where gad7_score > 10
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the gad7_score
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by level_of_education
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields using their `@id`.

Let's plot the distribution of GAD-7 scores and compare them by level of education.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for GAD-7 scores
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id} (GAD-7 Scores)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot comparing scores by education
if group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load Croissant schema metadata and records using `mlcroissant`.
- Reference all entities by their `@id`, as per FAIR standards.
- Examine demographics and survey scores (e.g., GAD-7) from a Kenyan mental health survey.
- Normalize scores and compare them across education levels.
- Visualize distributions to gain insights for AI/ML impact and public health policy.

This FAIR² dataset exemplifies structured, AI-ready data for global research and development.